# **Notebook Projet: Traduction Multilingue**

@Mathias Barrois

**Problématique :**

*"Comment traduire automatiquement des vidéos francophones en plusieurs
langues en combinant transcription automatique (ASR) et traduction neuronale (NMT), et quelles
architectures Deep Learning offrent le meilleur compromis qualité/robustesse/coût ?"*


<img 
  src="https://y.yarn.co/22d24c31-a73f-4374-93c9-069310624998_text.gif"
  width="300"
/>

**Source Data :** http://www.openslr.org/100/

### Strucuture du Notebook : 

- Config du Notebook
- Extraction ASR
- Différentes Approches de traduction
- Comparaisons entres approches
- Pipeline utiliser
- Conclusion


## **<u>1) Config du Notebook**


### **a) Pré-requis et structure attendue**

Data utilisé :
- `data/fr-fr/...` : corpus mTEDx français (audio, segments, textes)
- `doc/Projet_Traduction_Video_Multilingue_DL.pdf` : consignes du projet

In [ ]:
import glob
import os
import shutil
import site
from pathlib import Path

user_site = Path(site.getusersitepackages())
patterns = [
    "transformers", "transformers-*",
    "tokenizers", "tokenizers-*",
    "huggingface_hub", "huggingface_hub-*",
    "safetensors", "safetensors-*",
    "sentencepiece", "sentencepiece-*",
]

# Petit nettoyage de la libérie
for pattern in patterns:
    for path_str in glob.glob(str(user_site / pattern)):
        pth = Path(path_str)
        try:
            if pth.is_dir():
                shutil.rmtree(pth, ignore_errors=True)
            elif pth.exists():
                pth.unlink()
        except Exception:
            pass



### **b) Installation des librairies**


In [ ]:
# Reinstallation stable (apres nettoyage)
""" %pip install -U --no-cache-dir --force-reinstall \
  "transformers==4.48.2" "tokenizers==0.21.0" "sentencepiece>=0.1.99,<0.3" \
  "huggingface-hub>=0.24,<1.0" "safetensors>=0.4.3" \
  pandas numpy tqdm torch soundfile sacrebleu jiwer evaluate nltk psutil \
  matplotlib seaborn edge-tts nest-asyncio


  """


### **c) Config du Projet**

On centralise:
- chemins de données,
- split de travail,
- langues cibles,
- paramètres de reproductibilité (seed),
- reporting environnement (CPU/GPU/versions).


In [ ]:
from dataclasses import dataclass
from pathlib import Path
from typing import List
import os
import random
import platform

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm

# graine
SEED = 42

random.seed(SEED)

np.random.seed(SEED)

torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Racine du projet
PROJECT_ROOT = Path("..").resolve() if (Path("..").resolve() / "data").exists() else Path(".").resolve()

DATA_ROOT = PROJECT_ROOT / "data" / "fr-fr" / "data"
OUTPUT_ROOT = PROJECT_ROOT / "outputs"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

SPLIT = "test"  # "train", "valid" ou "test"
MAX_SEGMENTS = 80

# Source FR -> 4 langues cibles
SRC_LANG = "fr"
TARGET_LANGS = ["en"]#, "es", "pt", "it"]
TARGET_LANGS_MARIAN = TARGET_LANGS.copy()

# Pour accelerer: comparaisons limitees a fr->en
COMPARE_LANGS = ["en"]

# Reglages vitesse (augmente ces valeurs si tu veux des comparaisons plus larges)
COMPARE_MAX_SEGMENTS   = 64   # echantillon max pour eval externe
BENCH_MAX_SEGMENTS_CFG = 24 # echantillon max pour benchmark temps/ressources


# Approche locale LLM via Ollama (optionnelle)
ENABLE_OLLAMA   = True
OLLAMA_MODEL    = "gemma3:27b"
OLLAMA_BASE_URL = "http://192.168.1.152:11434"

OLLAMA_TIMEOUT_SEC = 120
# Limite optionnelle pour eviter des runs trop longs (None = tout)
OLLAMA_MAX_SEGMENTS = 24

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_ROOT:", DATA_ROOT)
print("OUTPUT_ROOT:", OUTPUT_ROOT)
print("SEED:", SEED)
print("Python:", platform.python_version())
print("Torch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("GPU: non detecte (execution CPU)")
print("ENABLE_OLLAMA:", ENABLE_OLLAMA, "| MODEL:", OLLAMA_MODEL)
print("COMPARE_LANGS:", COMPARE_LANGS)
print("COMPARE_MAX_SEGMENTS:", COMPARE_MAX_SEGMENTS)
print("BENCH_MAX_SEGMENTS_CFG:", BENCH_MAX_SEGMENTS_CFG)
print("OLLAMA_MAX_SEGMENTS:", OLLAMA_MAX_SEGMENTS)


### **d) Extraction de la Data**

Le corpus mTEDx fournit des fichiers parallèles ligne par ligne:
- `txt/segments` : identifiant segment + intervalle temporel
- `txt/<split>.fr` : transcription source française

On les fusionne dans un DataFrame unique pour préparer ASR et traduction.


In [ ]:
# Structure de la data
@dataclass
class Segment:
    segment_id: str
    talk_id: str
    start_sec: float
    end_sec: float
    text_src: str


def read_segments_file(path: Path) -> pd.DataFrame:
    """

    Lit chaque segements du fichier

    Args:
        path (Path): chemin du fichier

    Returns:
        pd.DataFrame: structure de données de sortie
    """

    rows = []
    
    with path.open("r", encoding="utf-8") as f:
        
        # Parcours chaque ligne
        for line in f:


            parts = line.strip().split()
            if len(parts) != 4:
                continue
            seg_id, talk_id, start_s, end_s = parts
            rows.append({
                "segment_id": seg_id,
                "talk_id": talk_id,
                "start_sec": float(start_s),
                "end_sec": float(end_s),
            })
    return pd.DataFrame(rows)


def read_text_file(path: Path) -> pd.Series:
    """

    Lit le texte dans le fichier

    Args:
        path (Path): _description_

    Returns:
        pd.Series: _description_
    """

    with path.open("r", encoding="utf-8") as f:
        lines = [ln.rstrip("") for ln in f]
    return pd.Series(lines, name="text_src")


def load_split_dataframe(data_root: Path, split: str, src_lang: str) -> pd.DataFrame:
    """

    Load dans un dataframe

    Args:
        data_root (Path): _description_
        split (str): _description_
        src_lang (str): _description_

    Returns:
        pd.DataFrame: _description_
    """
    
    txt_dir   = data_root / split / "txt"
    seg_path  = txt_dir / "segments"
    text_path = txt_dir / f"{split}.{src_lang}"

    df_seg  = read_segments_file(seg_path)
    sr_text = read_text_file(text_path)

    n  = min(len(df_seg), len(sr_text))
    df = df_seg.iloc[:n].copy()
    
    df["text_src"]     = sr_text.iloc[:n].values
    df["duration_sec"] = df["end_sec"] - df["start_sec"]
    df["split"]        = split

    return df


df_all = load_split_dataframe(DATA_ROOT, SPLIT, SRC_LANG)
print("Nombre total de segments:", len(df_all))
df_all.head(3)


### **e) Echantillonnage de travail**

On garde des sous ensembles pour par avoir trop de data
- soit les `MAX_SEGMENTS` premiers,
- soit un seul talk.


In [ ]:
# Si on veut tout les segments
# df_work = df_all.iloc[:MAX_SEGMENTS].copy()

# Option B: un seul talk pour une pipeline de bout en bout plus lisible
top_talk = df_all["talk_id"].value_counts().index[0]
df_work = df_all[df_all["talk_id"] == top_talk].copy()

df_work = df_work.iloc[:MAX_SEGMENTS].reset_index(drop=True)
print("Talk selectionne:", top_talk)
print("Segments retenus:", len(df_work))
print("Duree totale (min):", round(df_work["duration_sec"].sum() / 60, 2))
df_work.head(5)


### **f)  ASR: transcription**

Dans le dataset, l'audio est en `wav/*.flac` (un fichier par talk).
On applique un modèle ASR : **Whisper** pour obtenir une transcription temporelle en français.


In [ ]:
import torch
from transformers import pipeline

# Choix du modele ASR 
ASR_MODEL_NAME = "openai/whisper-medium" if torch.cuda.is_available() else "openai/whisper-small"
DEVICE         = 0 if torch.cuda.is_available() else -1

# Reglages ASR
ASR_CHUNK_LENGTH_S = 30
ASR_BATCH_SIZE = 8 if torch.cuda.is_available() else 2

print("GPU dispo :", torch.cuda.is_available())
print("Device pipeline :", DEVICE)
print("ASR model :", ASR_MODEL_NAME)

asr_pipe = pipeline(
    task="automatic-speech-recognition",
    model=ASR_MODEL_NAME,
    device=DEVICE,
)


In [ ]:
# Fichier audio principal du talk selectionne
talk_id = df_work.loc[0, "talk_id"]
audio_path = DATA_ROOT / SPLIT / "wav" / f"{talk_id}.flac"

print("Audio utilise:", audio_path)
assert audio_path.exists(), f"Audio introuvable: {audio_path}"

asr_result = asr_pipe(
    str(audio_path),
    return_timestamps=True,
    chunk_length_s=ASR_CHUNK_LENGTH_S,
    batch_size=ASR_BATCH_SIZE,
    generate_kwargs={
        "language": "french",
        "task": "transcribe",
        "temperature": 0.0,
    },
)

print("Cles ASR:", list(asr_result.keys()))
print("Apercu texte global:")
print(asr_result.get("text", "")[:500])


### **g) Conversion ASR en DataFrame composé de segments temporels**

On transforme la sortie brute ASR en table propre:
- `start_sec`, `end_sec`, `text_asr`
- utile pour exporter en SRT/VTT
- utile aussi pour la traduction ligne par ligne


In [ ]:
def asr_chunks_to_df(asr_output) -> pd.DataFrame:
    chunks = asr_output.get("chunks", [])
    rows = []
    for i, ch in enumerate(chunks):
        ts = ch.get("timestamp", (None, None))
        start_sec, end_sec = ts if isinstance(ts, (tuple, list)) and len(ts) == 2 else (None, None)
        rows.append({
            "idx": i,
            "start_sec": start_sec,
            "end_sec": end_sec,
            "text_asr": (ch.get("text", "") or "").strip(),
        })

    df = pd.DataFrame(rows)
    df = df.dropna(subset=["start_sec", "end_sec"]).copy()
    df = df[df["text_asr"].str.len() > 0].reset_index(drop=True)
    return df


df_asr = asr_chunks_to_df(asr_result)
print("Segments ASR exploitables:", len(df_asr))
df_asr.head(10)


### **h) Évaluation ASR (WER/CER) + analyse d'erreurss**

On calcule une première mesure globale WER/CER entre la transcription de référence FR (`df_work.text_src`) et la sortie ASR.
Puis on affiche quelques exemples pour analyse qualitative des erreurs ASR (noms propres, ponctuation, chiffres).


In [ ]:
from jiwer import wer, cer
import re
import unicodedata


def normalize_for_asr_eval(text: str) -> str:
    """Normalise le texte pour une comparaison ASR plus juste (ponctuation/casse/espaces)."""
    s = str(text or "").lower().strip()
    s = unicodedata.normalize("NFKD", s)
    s = "".join(ch for ch in s if not unicodedata.combining(ch))
    s = re.sub(r"[^a-z0-9\s]", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s


# Fenetre de reference (souvent tronquee par MAX_SEGMENTS): on evalue uniquement cette zone.
ref_start = float(df_work['start_sec'].min()) if len(df_work) else 0.0
ref_end = float(df_work['end_sec'].max()) if len(df_work) else 0.0

# Hypothese ASR alignee sur la meme fenetre temporelle.
if 'df_asr' in globals() and len(df_asr) > 0:
    asr_win = df_asr[(df_asr['start_sec'] < ref_end) & (df_asr['end_sec'] > ref_start)].copy()
    asr_win = asr_win.sort_values(['start_sec', 'end_sec']).reset_index(drop=True)
    hyp_text_raw = " ".join(asr_win['text_asr'].fillna('').tolist()).strip()
else:
    asr_win = None
    hyp_text_raw = asr_result.get('text', '').strip()

ref_text_raw = " ".join(df_work.sort_values(['start_sec', 'end_sec'])['text_src'].fillna('').tolist()).strip()

ref_text = normalize_for_asr_eval(ref_text_raw)
hyp_text = normalize_for_asr_eval(hyp_text_raw)

asr_wer = wer(ref_text, hyp_text) if ref_text and hyp_text else None
asr_cer = cer(ref_text, hyp_text) if ref_text and hyp_text else None

print(f"Fenetre comparee: [{ref_start:.2f}s -> {ref_end:.2f}s] | seg_ref={len(df_work)} | seg_asr={len(asr_win) if asr_win is not None else 'N/A'}")
print("Ref mots:", len(ref_text.split()), "| Hyp mots:", len(hyp_text.split()))
print("ASR WER:", round(asr_wer, 4) if asr_wer is not None else "N/A")
print("ASR CER:", round(asr_cer, 4) if asr_cer is not None else "N/A")

if asr_wer is not None and asr_wer > 1.0:
    print("[WARN] WER > 1.0: verifier l'alignement temporel ref/hyp ou la qualite ASR sur ce talk.")

# Petit debug lisible
debug_ref = df_work[['start_sec', 'end_sec', 'text_src']].head(5).copy()
if asr_win is not None and len(asr_win) > 0:
    debug_hyp = asr_win[['start_sec', 'end_sec', 'text_asr']].head(5).copy()
    display(debug_ref)
    display(debug_hyp)
else:
    display(debug_ref)


## **<u>2) Traduction via différentes approches</u>**

Ici on va utiliser et comparer trois approches de traductions automatiques :
- Approche A : MarianMT : Transformer
- Approche B : NLLB-200
- Approche C : Ollama via Gemma3:32b : Large Language Model
- Approche D : LSTM entrainé

### **a) Approche A : MarianMT**

**Modèle utilisé:** MarianMT (famille Transformer encoder-decoder), avec un modèle par paire `fr->langue`.

**Pourquoi cette approche:**
- modèle spécialisé bilingue, souvent plus précis sur une paire donnée,
- plus léger qu'un très gros modèle multilingue,
- rapide pour l'inférence sur un volume modéré.

**Limites:**
- il faut charger un modèle différent pour chaque langue cible,
- couverture variable selon les paires,
- robustesse plus faible hors domaine.

**En résumé:** Marian sert de baseline Transformer spécialisée (qualité/latence souvent bonnes sur langues proches).


In [ ]:
import importlib

MARIAN_MODEL_CANDIDATES = {
    "en": ["Helsinki-NLP/opus-mt-fr-en"],
    "es": ["Helsinki-NLP/opus-mt-fr-es"],
    "it": ["Helsinki-NLP/opus-mt-fr-it"],
    "pt": ["Helsinki-NLP/opus-mt-fr-pt"],
}

MARIAN_PIVOT_CANDIDATES = {
    "pt": [
        ("Helsinki-NLP/opus-mt-fr-en", "Helsinki-NLP/opus-mt-en-pt"),
        ("Helsinki-NLP/opus-mt-fr-es", "Helsinki-NLP/opus-mt-es-pt"),
    ]
}

_MARIAN_MODEL_CACHE = {}


def normalize_lang_tag(lang: str) -> str:
    return lang.lower().split("-")[0]


def ensure_sentencepiece() -> None:
    try:
        importlib.import_module("sentencepiece")
    except ModuleNotFoundError as e:
        raise RuntimeError(
            "`sentencepiece` est absent. Execute la cellule d'installation (%pip ...), puis redemarre le kernel."
        ) from e


def _load_marian_model(model_name: str):
    """Charge et met en cache un modele Marian pour eviter les reloads."""
    if model_name in _MARIAN_MODEL_CACHE:
        return _MARIAN_MODEL_CACHE[model_name]

    from transformers import MarianMTModel, MarianTokenizer

    tokenizer = MarianTokenizer.from_pretrained(model_name)
    model = MarianMTModel.from_pretrained(model_name)
    model.to("cuda" if torch.cuda.is_available() else "cpu")
    model.eval()

    _MARIAN_MODEL_CACHE[model_name] = (tokenizer, model)
    return tokenizer, model


def _translate_with_model_name(texts: List[str], model_name: str, batch_size: int, desc: str) -> List[str]:
    tokenizer, model = _load_marian_model(model_name)
    outputs = []
    for i in tqdm(range(0, len(texts), batch_size), desc=desc):
        batch = texts[i:i + batch_size]
        enc = tokenizer(batch, return_tensors="pt", padding=True, truncation=True)
        enc = {k: v.to(model.device) for k, v in enc.items()}
        with torch.no_grad():
            gen = model.generate(**enc, max_new_tokens=128)
        outputs.extend(tokenizer.batch_decode(gen, skip_special_tokens=True))
    return outputs


def translate_with_marian(texts: List[str], target_lang: str, batch_size: int = 16) -> List[str]:
    ensure_sentencepiece()
    target_key = normalize_lang_tag(target_lang)

    last_err = None
    for model_name in MARIAN_MODEL_CANDIDATES.get(target_key, []):
        try:
            return _translate_with_model_name(texts, model_name, batch_size, f"Marian {SRC_LANG}->{target_lang}")
        except Exception as e:
            last_err = e

    for first_model, second_model in MARIAN_PIVOT_CANDIDATES.get(target_key, []):
        try:
            pivot_texts = _translate_with_model_name(texts, first_model, batch_size, f"Marian pivot-1 ({first_model.split('/')[-1]})")
            return _translate_with_model_name(pivot_texts, second_model, batch_size, f"Marian pivot-2 ({second_model.split('/')[-1]})")
        except Exception as e:
            last_err = e

    if "translate_with_nllb" in globals():
        print(f"[WARN] Marian indisponible pour {target_lang}; fallback NLLB.")
        return translate_with_nllb(texts, src_lang=SRC_LANG, tgt_lang=target_lang)

    raise RuntimeError(f"Impossible de traduire en {target_lang}. Derniere erreur: {last_err}")


In [ ]:
# On traduit les segments ASR
texts_src = df_asr["text_asr"].fillna("").tolist()

marian_failures = {}
for lang in TARGET_LANGS_MARIAN:
    col = f"text_marian_{lang}"
    try:
        df_asr[col] = translate_with_marian(texts_src, target_lang=lang)
    except Exception as e:
        marian_failures[lang] = str(e)
        print(f"[WARN] Marian indisponible pour {lang}: {e}")
        # Fallback minimal pour conserver une colonne exploitable
        df_asr[col] = texts_src

# S'assure que toutes les langues cibles ont une colonne Marian, meme si non traitees
for lang in TARGET_LANGS:
    col = f"text_marian_{lang}"
    if col not in df_asr.columns:
        df_asr[col] = texts_src

if marian_failures:
    print("[INFO] Echecs Marian:")
    for lang, err in marian_failures.items():
        print(f" - {lang}: {err}")

df_asr.head(8)


### **b)  Approche B : NLLB-200**

**Modele utilise:** `facebook/nllb-200-distilled-600M` (modele massivement multilingue).

**Pourquoi cette approche:**
- un seul modele pour de nombreuses langues,
- coherence de pipeline multi-langues,
- bonne robustesse quand la paire est moins bien couverte par les modeles specialises.

**Limites:**
- modele plus lourd (memoire/latence),
- qualite parfois inferieure a un tres bon modele specialise sur certaines paires,
- cout d'inference plus eleve.

**Differences cles vs Marian:**
- Marian: specialise par paire, souvent rapide et precis sur paires couvertes.
- NLLB: multilingue natif, couverture large.



### **c)  Approche C : Ollama local**

**Modele utilise:** `gemma3:27b` 

**Pourquoi cette approche:**
- baseline LLM locale, sans API cloud,
- flexible par prompting (peut etre adapte au domaine),
- utile pour comparer un LLM generaliste face a des modeles MT dedies.

**Limites:**
- souvent plus lent en inference,
- format de sortie moins stable sans contraintes strictes,
- qualite variable selon langue/domaine.

**Positionnement:**
- Marian/NLLB restent les approches MT principales.
- Ollama est une comparaison supplementaire "Approche C" dans les tableaux/benchmarks.


In [ ]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
import json
import urllib.request
import urllib.error

NLLB_MODEL_NAME = "facebook/nllb-200-distilled-600M"

NLLB_LANG = {
    "fr": "fra_Latn",
    "en": "eng_Latn",
    "es": "spa_Latn",
    "pt": "por_Latn",
    "it": "ita_Latn",
}

LANG_NAME = {
    "fr": "francais",
    "en": "anglais",
    "es": "espagnol",
    "pt": "portugais",
    "it": "italien",
}

nllb_tokenizer = AutoTokenizer.from_pretrained(NLLB_MODEL_NAME)
nllb_model = AutoModelForSeq2SeqLM.from_pretrained(NLLB_MODEL_NAME)
nllb_model.to("cuda" if torch.cuda.is_available() else "cpu")


def translate_with_nllb(texts: List[str], src_lang: str, tgt_lang: str, batch_size: int = 16) -> List[str]:
    # Traduction batch par batch avec forçage de la langue cible.
    src_key = normalize_lang_tag(src_lang)
    tgt_key = normalize_lang_tag(tgt_lang)
    if src_key not in NLLB_LANG or tgt_key not in NLLB_LANG:
        raise ValueError(f"Code NLLB manquant pour {src_lang}->{tgt_lang} ({src_key}->{tgt_key})")

    src_code = NLLB_LANG[src_key]
    tgt_code = NLLB_LANG[tgt_key]

    nllb_tokenizer.src_lang = src_code
    forced_bos_id = nllb_tokenizer.convert_tokens_to_ids(tgt_code)

    outputs = []
    for i in tqdm(range(0, len(texts), batch_size), desc=f"NLLB {src_lang}->{tgt_lang}"):
        batch = texts[i:i + batch_size]
        enc = nllb_tokenizer(batch, return_tensors="pt", padding=True, truncation=True)
        enc = {k: v.to(nllb_model.device) for k, v in enc.items()}
        with torch.no_grad():
            gen = nllb_model.generate(
                **enc,
                forced_bos_token_id=forced_bos_id,
                max_new_tokens=128,
            )
        outputs.extend(nllb_tokenizer.batch_decode(gen, skip_special_tokens=True))

    return outputs


def ollama_is_available(base_url: str = OLLAMA_BASE_URL) -> bool:
    # Vérifie rapidement que le serveur Ollama répond.
    try:
        req = urllib.request.Request(f"{base_url}/api/tags", method="GET")
        with urllib.request.urlopen(req, timeout=3) as resp:
            return 200 <= resp.status < 300
    except Exception:
        return False


def _ollama_prompt(text: str, src_lang: str, tgt_lang: str) -> str:
    # Prompt volontairement contraint pour limiter les sorties hors-format.
    src_name = LANG_NAME.get(normalize_lang_tag(src_lang), src_lang)
    tgt_name = LANG_NAME.get(normalize_lang_tag(tgt_lang), tgt_lang)
    return (
        f"Traduis du {src_name} vers le {tgt_name}. "
        "Retourne uniquement la traduction, sans commentaire ni guillemets.\n"
        f"Texte: {text}"
    )


def _ollama_generate(prompt: str, model_name: str, base_url: str) -> str:
    # Appel HTTP direct à /api/generate en mode non-stream.
    payload = {
        "model": model_name,
        "prompt": prompt,
        "stream": False,
        "options": {
            "temperature": 0,
        },
    }
    data = json.dumps(payload).encode("utf-8")
    req = urllib.request.Request(
        f"{base_url}/api/generate",
        data=data,
        headers={"Content-Type": "application/json"},
        method="POST",
    )
    with urllib.request.urlopen(req, timeout=OLLAMA_TIMEOUT_SEC) as resp:
        body = resp.read().decode("utf-8")
    out = json.loads(body).get("response", "").strip()
    return out.strip('"').strip()


def translate_with_ollama(
    texts: List[str],
    src_lang: str,
    tgt_lang: str,
    model_name: str = OLLAMA_MODEL,
    base_url: str = OLLAMA_BASE_URL,
) -> List[str]:
    # Wrapper global utilisé dans les comparaisons (qualité + temps).
    if not ollama_is_available(base_url):
        raise RuntimeError(f"Ollama indisponible sur {base_url}. Lance 'ollama serve'.")

    outputs = []
    for txt in tqdm(texts, desc=f"Ollama {model_name} {src_lang}->{tgt_lang}"):
        txt = (txt or "").strip()
        if not txt:
            outputs.append("")
            continue
        prompt = _ollama_prompt(txt, src_lang, tgt_lang)
        try:
            outputs.append(_ollama_generate(prompt, model_name=model_name, base_url=base_url))
        except urllib.error.HTTPError as e:
            raise RuntimeError(f"Ollama HTTP {e.code}: {e.reason}") from e
        except Exception as e:
            raise RuntimeError(f"Echec Ollama: {e}") from e

    return outputs


In [ ]:
for lang in TARGET_LANGS:
    col = f"text_nllb_{lang}"
    df_asr[col] = translate_with_nllb(texts_src, src_lang=SRC_LANG, tgt_lang=lang)

# Approche C optionnelle: Ollama local (gemma3:27b)
ollama_failures = {}
if ENABLE_OLLAMA:
    for lang in TARGET_LANGS:
        col = f"text_ollama_{lang}"
        n = len(texts_src) if OLLAMA_MAX_SEGMENTS is None else min(len(texts_src), int(OLLAMA_MAX_SEGMENTS))
        try:
            out = translate_with_ollama(texts_src[:n], src_lang=SRC_LANG, tgt_lang=lang)
            if n < len(texts_src):
                out = out + [""] * (len(texts_src) - n)
            df_asr[col] = out
        except Exception as e:
            ollama_failures[lang] = str(e)
            print(f"[WARN] Ollama indisponible pour {lang}: {e}")
            df_asr[col] = [""] * len(texts_src)
else:
    for lang in TARGET_LANGS:
        df_asr[f"text_ollama_{lang}"] = [""] * len(texts_src)

if ollama_failures:
    print("[INFO] Echecs Ollama:")
    for lang, err in ollama_failures.items():
        print(f" - {lang}: {err}")

# Affichage lisible: on montre les comparaisons utiles, sans lignes Ollama vides.
lang_show = COMPARE_LANGS[0] if len(COMPARE_LANGS) else TARGET_LANGS[0]
base_cols = ["start_sec", "end_sec", "text_asr", f"text_marian_{lang_show}", f"text_nllb_{lang_show}"]
if f"text_lstm_en" in df_asr.columns and lang_show == "en":
    base_cols.append("text_lstm_en")

print("Apercu comparaison (Marian/NLLB/LSTM):")
display(df_asr[base_cols].head(8))

oll_col = f"text_ollama_{lang_show}"
if oll_col in df_asr.columns:
    mask_oll = df_asr[oll_col].fillna("").str.strip().str.len() > 0
    n_non_empty = int(mask_oll.sum())
    n_total = len(df_asr)
    print(f"Ollama non vide: {n_non_empty}/{n_total} segments ({(100*n_non_empty/max(1,n_total)):.1f}%)")
    if n_non_empty > 0:
        show_cols = ["start_sec", "end_sec", "text_asr", f"text_marian_{lang_show}", f"text_nllb_{lang_show}", oll_col]
        print("Apercu Ollama (lignes non vides uniquement):")
        display(df_asr.loc[mask_oll, show_cols].head(8))
    else:
        print("Aucune ligne Ollama non vide a afficher.")


### **d) Approche D : Seq2Seq LSTM (FR->EN) sur `data/fr-en`**

Section minimale pour couvrir l'approche 1 du PDF (baseline séquentielle):
- chargement des paires parallèles FR/EN,
- tokenisation simple whitespace,
- vocabulaire + DataLoader,
- modèle Encoder-Decoder LSTM,
- entraînement et inférence greedy,
- évaluation BLEU/chrF sur le split `test`.

Note: c'est un baseline pédagogique compact (pas SOTA).


In [ ]:
from collections import Counter
from dataclasses import dataclass
from pathlib import Path
from typing import List, Dict, Tuple

import random
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sacrebleu import corpus_bleu, corpus_chrf

PARA_ROOT = PROJECT_ROOT / 'data' / 'fr-en' / 'data'
assert PARA_ROOT.exists(), f"Dossier fr-en introuvable: {PARA_ROOT}"

SPECIALS = ['<pad>', '<bos>', '<eos>', '<unk>']
PAD_IDX, BOS_IDX, EOS_IDX, UNK_IDX = 0, 1, 2, 3


def simple_tok(text: str) -> List[str]:
    return str(text).strip().lower().split()


def load_parallel_split(root: Path, split: str = 'train') -> List[Tuple[str, str]]:
    txt = root / split / 'txt'
    fr_path = txt / f'{split}.fr'
    en_path = txt / f'{split}.en'
    fr_lines = [ln.strip() for ln in fr_path.read_text(encoding='utf-8').splitlines()]
    en_lines = [ln.strip() for ln in en_path.read_text(encoding='utf-8').splitlines()]
    n = min(len(fr_lines), len(en_lines))
    return [(fr_lines[i], en_lines[i]) for i in range(n)]


def build_vocab(texts: List[str], min_freq: int = 2, max_size: int = 30000) -> Dict[str, int]:
    cnt = Counter()
    for t in texts:
        cnt.update(simple_tok(t))
    vocab = {tok: i for i, tok in enumerate(SPECIALS)}
    for tok, freq in cnt.most_common(max_size):
        if freq < min_freq:
            continue
        if tok not in vocab:
            vocab[tok] = len(vocab)
    return vocab


def ids_from_text(text: str, vocab: Dict[str, int], add_bos=False, add_eos=False) -> List[int]:
    ids = [vocab.get(tok, UNK_IDX) for tok in simple_tok(text)]
    if add_bos:
        ids = [BOS_IDX] + ids
    if add_eos:
        ids = ids + [EOS_IDX]
    return ids


def text_from_ids(ids: List[int], inv_vocab: Dict[int, str]) -> str:
    toks = []
    for i in ids:
        if i in (PAD_IDX, BOS_IDX):
            continue
        if i == EOS_IDX:
            break
        toks.append(inv_vocab.get(i, '<unk>'))
    return ' '.join(toks)


In [ ]:
class ParallelDataset(Dataset):
    def __init__(self, pairs, src_vocab, tgt_vocab):
        """Fonction utilitaire pour   init  ."""
        self.src = [ids_from_text(fr, src_vocab, add_bos=True, add_eos=True) for fr, _ in pairs]
        self.tgt = [ids_from_text(en, tgt_vocab, add_bos=True, add_eos=True) for _, en in pairs]

    def __len__(self):
        """Fonction utilitaire pour   len  ."""
        return len(self.src)

    def __getitem__(self, idx):
        """Fonction utilitaire pour   getitem  ."""
        return torch.tensor(self.src[idx], dtype=torch.long), torch.tensor(self.tgt[idx], dtype=torch.long)


def collate_pad(batch):
    """Pad les sequences d un batch pour le DataLoader."""
    srcs, tgts = zip(*batch)
    src_len = max(x.size(0) for x in srcs)
    tgt_len = max(x.size(0) for x in tgts)
    src_pad = torch.full((len(batch), src_len), PAD_IDX, dtype=torch.long)
    tgt_pad = torch.full((len(batch), tgt_len), PAD_IDX, dtype=torch.long)
    for i, (s, t) in enumerate(batch):
        src_pad[i, :s.size(0)] = s
        tgt_pad[i, :t.size(0)] = t
    return src_pad, tgt_pad


class Encoder(nn.Module):
    def __init__(self, vocab_size, emb_dim=256, hid_dim=512, n_layers=1, dropout=0.2):
        """Fonction utilitaire pour   init  ."""
        super().__init__()
        self.emb = nn.Embedding(vocab_size, emb_dim, padding_idx=PAD_IDX)
        self.rnn = nn.LSTM(emb_dim, hid_dim, num_layers=n_layers, batch_first=True, dropout=dropout if n_layers > 1 else 0.0)

    def forward(self, src):
        """Fonction utilitaire pour forward."""
        emb = self.emb(src)
        _, (h, c) = self.rnn(emb)
        return h, c


class Decoder(nn.Module):
    def __init__(self, vocab_size, emb_dim=256, hid_dim=512, n_layers=1, dropout=0.2):
        """Fonction utilitaire pour   init  ."""
        super().__init__()
        self.emb = nn.Embedding(vocab_size, emb_dim, padding_idx=PAD_IDX)
        self.rnn = nn.LSTM(emb_dim, hid_dim, num_layers=n_layers, batch_first=True, dropout=dropout if n_layers > 1 else 0.0)
        self.out = nn.Linear(hid_dim, vocab_size)

    def forward(self, inp_tok, h, c):
        """Fonction utilitaire pour forward."""
        # inp_tok: [B]
        emb = self.emb(inp_tok.unsqueeze(1))
        o, (h, c) = self.rnn(emb, (h, c))
        logits = self.out(o.squeeze(1))
        return logits, h, c


class Seq2SeqLSTM(nn.Module):
    def __init__(self, encoder, decoder, device):
        """Fonction utilitaire pour   init  ."""
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.device = device

    def forward(self, src, tgt, teacher_forcing=0.5):
        """Fonction utilitaire pour forward."""
        # src:[B,S], tgt:[B,T]
        bsz, tgt_len = tgt.size()
        vocab_size = self.decoder.out.out_features
        outputs = torch.zeros(bsz, tgt_len, vocab_size, device=self.device)

        h, c = self.encoder(src)
        inp = tgt[:, 0]  # <bos>
        for t in range(1, tgt_len):
            logits, h, c = self.decoder(inp, h, c)
            outputs[:, t] = logits
            top1 = logits.argmax(dim=1)
            use_tf = random.random() < teacher_forcing
            inp = tgt[:, t] if use_tf else top1
        return outputs


@torch.no_grad()
def greedy_decode(model, src_tensor, max_len=80):
    """Decode pas a pas sans beam search pour rester simple."""
    model.eval()
    h, c = model.encoder(src_tensor)
    inp = torch.full((src_tensor.size(0),), BOS_IDX, dtype=torch.long, device=src_tensor.device)
    preds = []
    for _ in range(max_len):
        logits, h, c = model.decoder(inp, h, c)
        nxt = logits.argmax(dim=1)
        preds.append(nxt)
        inp = nxt
    return torch.stack(preds, dim=1)  # [B,L]


In [ ]:
@dataclass
class LSTMRun:
    train_pairs: int = 12000
    valid_pairs: int = 1200
    test_pairs: int = 1200
    batch_size: int = 64
    emb_dim: int = 256
    hid_dim: int = 512
    n_layers: int = 1
    epochs: int = 100
    lr: float = 2e-3
    min_freq: int = 2
    max_vocab: int = 30000

cfg = LSTMRun()

pairs_train = load_parallel_split(PARA_ROOT, 'train')[:cfg.train_pairs]
pairs_valid = load_parallel_split(PARA_ROOT, 'valid')[:cfg.valid_pairs]
pairs_test = load_parallel_split(PARA_ROOT, 'test')[:cfg.test_pairs]

src_vocab = build_vocab([x for x, _ in pairs_train], min_freq=cfg.min_freq, max_size=cfg.max_vocab)
tgt_vocab = build_vocab([y for _, y in pairs_train], min_freq=cfg.min_freq, max_size=cfg.max_vocab)
inv_tgt_vocab = {v: k for k, v in tgt_vocab.items()}

train_ds = ParallelDataset(pairs_train, src_vocab, tgt_vocab)
valid_ds = ParallelDataset(pairs_valid, src_vocab, tgt_vocab)
test_ds = ParallelDataset(pairs_test, src_vocab, tgt_vocab)

train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True, collate_fn=collate_pad)
valid_loader = DataLoader(valid_ds, batch_size=cfg.batch_size, shuffle=False, collate_fn=collate_pad)
test_loader = DataLoader(test_ds, batch_size=cfg.batch_size, shuffle=False, collate_fn=collate_pad)

lstm_device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
enc = Encoder(len(src_vocab), emb_dim=cfg.emb_dim, hid_dim=cfg.hid_dim, n_layers=cfg.n_layers)
dec = Decoder(len(tgt_vocab), emb_dim=cfg.emb_dim, hid_dim=cfg.hid_dim, n_layers=cfg.n_layers)
lstm_model = Seq2SeqLSTM(enc, dec, lstm_device).to(lstm_device)

criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX)
optim = torch.optim.Adam(lstm_model.parameters(), lr=cfg.lr)


def run_epoch(model, loader, train=True):
    """Execute une passe train/valid et retourne la loss moyenne."""
    # Boucle d'entrainement/validation standard.
    model.train() if train else model.eval()
    total_loss = 0.0
    for src, tgt in loader:
        src, tgt = src.to(lstm_device), tgt.to(lstm_device)
        if train:
            optim.zero_grad()
        out = model(src, tgt, teacher_forcing=0.5 if train else 0.0)
        logits = out[:, 1:].reshape(-1, out.size(-1))
        gold = tgt[:, 1:].reshape(-1)
        loss = criterion(logits, gold)
        if train:
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optim.step()
        total_loss += float(loss.item())
    return total_loss / max(1, len(loader))

for ep in range(1, cfg.epochs + 1):
    tr = run_epoch(lstm_model, train_loader, train=True)
    va = run_epoch(lstm_model, valid_loader, train=False)
    print(f"Epoch {ep}/{cfg.epochs} | train_loss={tr:.4f} | valid_loss={va:.4f}")


@torch.no_grad()
def eval_lstm_bleu_chrf(model, loader, inv_tgt_vocab):
    """Calcule BLEU/chrF du LSTM sur le loader fourni."""
    # Evaluation intrinsèque sur le split test fr-en.
    model.eval()
    hyps, refs = [], []
    for src, tgt in loader:
        src, tgt = src.to(lstm_device), tgt.to(lstm_device)
        pred = greedy_decode(model, src, max_len=tgt.size(1))
        for i in range(pred.size(0)):
            hyp = text_from_ids(pred[i].tolist(), inv_tgt_vocab)
            ref = text_from_ids(tgt[i].tolist(), inv_tgt_vocab)
            hyps.append(hyp)
            refs.append(ref)
    bleu = corpus_bleu(hyps, [refs]).score
    chrf = corpus_chrf(hyps, [refs]).score
    return bleu, chrf, hyps, refs


@torch.no_grad()
def translate_with_lstm(texts: List[str], max_len: int = 128) -> List[str]:
    # Inference utilitaire FR->EN pour comparer LSTM avec Marian/NLLB/Ollama.
    lstm_model.eval()
    outs = []
    for txt in texts:
        src_ids = ids_from_text(str(txt), src_vocab, add_bos=True, add_eos=True)
        src_t = torch.tensor(src_ids, dtype=torch.long, device=lstm_device).unsqueeze(0)
        pred_ids = greedy_decode(lstm_model, src_t, max_len=max_len)[0].tolist()
        outs.append(text_from_ids(pred_ids, inv_tgt_vocab))
    return outs


bleu_lstm, chrf_lstm, lstm_hyps, lstm_refs = eval_lstm_bleu_chrf(lstm_model, test_loader, inv_tgt_vocab)
print(f"LSTM BLEU test: {bleu_lstm:.2f}")
print(f"LSTM chrF test: {chrf_lstm:.2f}")

# Exemple utilitaire simple
sample_src = pairs_test[0][0]
src_ids = ids_from_text(sample_src, src_vocab, add_bos=True, add_eos=True)
src_t = torch.tensor(src_ids, dtype=torch.long, device=lstm_device).unsqueeze(0)
pred_ids = greedy_decode(lstm_model, src_t, max_len=80)[0].tolist()
print('FR source:', sample_src)
print('EN LSTM  :', text_from_ids(pred_ids, inv_tgt_vocab))

# Option pratique: ajoute la sortie LSTM EN dans df_asr pour comparaison qualitative.
if 'df_asr' in globals() and 'text_asr' in df_asr.columns:
    try:
        df_asr['text_lstm_en'] = translate_with_lstm(df_asr['text_asr'].fillna('').tolist())
    except Exception as e:
        print(f"[WARN] text_lstm_en non calcule: {e}")


### **e) Export sous-titres**

On exporte, pour chaque langue cible:
- un SRT Marian
- un SRT NLLB


In [ ]:
# Utilitaires d'export SRT (redefines sans risque si deja charges).
def _fmt_srt_time(sec: float) -> str:
    sec = max(0.0, float(sec))
    h = int(sec // 3600)
    m = int((sec % 3600) // 60)
    s = int(sec % 60)
    ms = int(round((sec - int(sec)) * 1000))
    if ms == 1000:
        s += 1
        ms = 0
    return f"{h:02}:{m:02}:{s:02},{ms:03}"


def export_srt(df, out_path, text_col='text_asr'):
    """Exporte un DataFrame de segments au format SRT."""
    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    lines = []
    for i, row in df.reset_index(drop=True).iterrows():
        start = _fmt_srt_time(row['start_sec'])
        end = _fmt_srt_time(row['end_sec'])
        txt = str(row.get(text_col, '') or '').strip()
        lines.append(str(i + 1))
        lines.append(f"{start} --> {end}")
        lines.append(txt)
        lines.append('')
    out_path.write_text("\n".join(lines), encoding='utf-8')


for lang in TARGET_LANGS:
    srt_marian = OUTPUT_ROOT / f"{talk_id}.marian.{lang}.srt"
    srt_nllb = OUTPUT_ROOT / f"{talk_id}.nllb.{lang}.srt"

    export_srt(df_asr, srt_marian, text_col=f"text_marian_{lang}")
    export_srt(df_asr, srt_nllb, text_col=f"text_nllb_{lang}")

    col_ollama = f"text_ollama_{lang}"
    if col_ollama in df_asr.columns and df_asr[col_ollama].fillna("").str.len().sum() > 0:
        srt_ollama = OUTPUT_ROOT / f"{talk_id}.ollama.{lang}.srt"
        export_srt(df_asr, srt_ollama, text_col=col_ollama)

print("Exports generes dans:", OUTPUT_ROOT)
for p in sorted(OUTPUT_ROOT.glob(f"{talk_id}*.srt"))[:40]:
    print("-", p.name)


## **<u>3) Evaluation et comparaison des différentes approches</u>**

Cette section active les metriques si des references paralleles existent localement.

Chemins detectes automatiquement (priorite):
- `data/parallel/fr-<lang>/<split>.ref.<lang>`
- `data/fr-<lang>/data/<split>/txt/<split>.<lang>`
- `data/mtedx_fr-<lang>/data/<split>/txt/<split>.<lang>`


### **a) Comparaison via BLEU / chrF / METEOR / COMET**


In [ ]:
from pathlib import Path
import pandas as pd
from sacrebleu import corpus_bleu, corpus_chrf

try:
    import evaluate
except Exception:
    evaluate = None


def find_parallel_files(project_root: Path, lang: str, split: str):
    """Cherche automatiquement une paire source/reference disponible."""
    candidates = [
        (
            project_root / 'data' / f'fr-{lang}' / 'data' / split / 'txt' / f'{split}.fr',
            project_root / 'data' / f'fr-{lang}' / 'data' / split / 'txt' / f'{split}.{lang}',
        ),
        (
            project_root / 'data' / f'mtedx_fr-{lang}' / 'data' / split / 'txt' / f'{split}.fr',
            project_root / 'data' / f'mtedx_fr-{lang}' / 'data' / split / 'txt' / f'{split}.{lang}',
        ),
        (
            project_root / 'data' / 'parallel' / f'fr-{lang}' / f'{split}.src.fr',
            project_root / 'data' / 'parallel' / f'fr-{lang}' / f'{split}.ref.{lang}',
        ),
    ]
    for src_p, ref_p in candidates:
        if src_p.exists() and ref_p.exists():
            return src_p, ref_p
    return None, None


rows = []
for lang in COMPARE_LANGS:
    src_path, ref_path = find_parallel_files(PROJECT_ROOT, lang, SPLIT)
    if src_path is None:
        rows.append({'lang': lang, 'status': 'skip', 'detail': 'reference absente'})
        continue

    srcs = [ln.strip() for ln in src_path.read_text(encoding='utf-8', errors='ignore').splitlines() if ln.strip()]
    refs = [ln.strip() for ln in ref_path.read_text(encoding='utf-8', errors='ignore').splitlines() if ln.strip()]
    n = min(len(srcs), len(refs), int(COMPARE_MAX_SEGMENTS))
    if n == 0:
        rows.append({'lang': lang, 'status': 'skip', 'detail': f'fichier vide: {ref_path}'})
        continue

    srcs, refs = srcs[:n], refs[:n]

    marian_err = ''
    nllb_err = ''
    ollama_err = ''
    lstm_err = ''

    try:
        hyp_marian = translate_with_marian(srcs, target_lang=lang)
    except Exception as e:
        hyp_marian = [''] * n
        marian_err = str(e)

    try:
        hyp_nllb = translate_with_nllb(srcs, src_lang='fr', tgt_lang=lang)
    except Exception as e:
        hyp_nllb = [''] * n
        nllb_err = str(e)

    hyp_ollama = [''] * n
    if ENABLE_OLLAMA:
        try:
            hyp_ollama = translate_with_ollama(srcs, src_lang='fr', tgt_lang=lang)
        except Exception as e:
            ollama_err = str(e)

    hyp_lstm = [''] * n
    if lang == 'en' and 'translate_with_lstm' in globals():
        try:
            hyp_lstm = translate_with_lstm(srcs)
        except Exception as e:
            lstm_err = str(e)

    bleu_marian = corpus_bleu(hyp_marian, [refs]).score
    bleu_nllb = corpus_bleu(hyp_nllb, [refs]).score
    chrf_marian = corpus_chrf(hyp_marian, [refs]).score
    chrf_nllb = corpus_chrf(hyp_nllb, [refs]).score

    bleu_ollama = corpus_bleu(hyp_ollama, [refs]).score if any(hyp_ollama) else None
    chrf_ollama = corpus_chrf(hyp_ollama, [refs]).score if any(hyp_ollama) else None
    bleu_lstm = corpus_bleu(hyp_lstm, [refs]).score if any(hyp_lstm) else None
    chrf_lstm = corpus_chrf(hyp_lstm, [refs]).score if any(hyp_lstm) else None

    meteor_marian = meteor_nllb = meteor_ollama = meteor_lstm = None
    comet_marian = comet_nllb = comet_ollama = comet_lstm = None
    detail_notes = []

    if evaluate is None:
        detail_notes.append('evaluate non installe -> METEOR/COMET non calcules')
    else:
        try:
            meteor = evaluate.load('meteor')
            meteor_marian = meteor.compute(predictions=hyp_marian, references=refs)['meteor']
            meteor_nllb = meteor.compute(predictions=hyp_nllb, references=refs)['meteor']
            if any(hyp_ollama):
                meteor_ollama = meteor.compute(predictions=hyp_ollama, references=refs)['meteor']
            if any(hyp_lstm):
                meteor_lstm = meteor.compute(predictions=hyp_lstm, references=refs)['meteor']
        except Exception:
            detail_notes.append('METEOR indisponible')

        try:
            comet = evaluate.load('comet')
            comet_marian = comet.compute(predictions=hyp_marian, references=refs, sources=srcs).get('mean_score')
            comet_nllb = comet.compute(predictions=hyp_nllb, references=refs, sources=srcs).get('mean_score')
            if any(hyp_ollama):
                comet_ollama = comet.compute(predictions=hyp_ollama, references=refs, sources=srcs).get('mean_score')
            if any(hyp_lstm):
                comet_lstm = comet.compute(predictions=hyp_lstm, references=refs, sources=srcs).get('mean_score')
        except Exception:
            detail_notes.append('COMET indisponible')

    if ENABLE_OLLAMA and ollama_err:
        detail_notes.append('Ollama indisponible pour cette langue')
    if lang == 'en' and 'translate_with_lstm' not in globals():
        detail_notes.append('LSTM non execute dans cette session')

    rows.append({
        'lang': lang,
        'status': 'ok',
        'src_path': str(src_path),
        'ref_path': str(ref_path),
        'n_segments': n,
        'bleu_marian': round(bleu_marian, 2),
        'bleu_nllb': round(bleu_nllb, 2),
        'bleu_ollama': round(bleu_ollama, 2) if bleu_ollama is not None else None,
        'bleu_lstm': round(bleu_lstm, 2) if bleu_lstm is not None else None,
        'chrf_marian': round(chrf_marian, 2),
        'chrf_nllb': round(chrf_nllb, 2),
        'chrf_ollama': round(chrf_ollama, 2) if chrf_ollama is not None else None,
        'chrf_lstm': round(chrf_lstm, 2) if chrf_lstm is not None else None,
        'meteor_marian': round(meteor_marian, 4) if meteor_marian is not None else None,
        'meteor_nllb': round(meteor_nllb, 4) if meteor_nllb is not None else None,
        'meteor_ollama': round(meteor_ollama, 4) if meteor_ollama is not None else None,
        'meteor_lstm': round(meteor_lstm, 4) if meteor_lstm is not None else None,
        'comet_marian': round(comet_marian, 4) if comet_marian is not None else None,
        'comet_nllb': round(comet_nllb, 4) if comet_nllb is not None else None,
        'comet_ollama': round(comet_ollama, 4) if comet_ollama is not None else None,
        'comet_lstm': round(comet_lstm, 4) if comet_lstm is not None else None,
        'marian_err': marian_err,
        'nllb_err': nllb_err,
        'ollama_err': ollama_err,
        'lstm_err': lstm_err,
        'detail': '; '.join(detail_notes) if detail_notes else 'ok',
    })

eval_translation_df = pd.DataFrame(rows)
eval_translation_df


### **b) Interprétation des métriques**

- `BLEU` : recouvrement en n-grammes avec la référence (plus haut = mieux).
- `chrF` : comparaison au niveau caractères (robuste sur phrases courtes).


Pas utilis
- `METEOR` : inclut variations lexicales (stemming/synonymes), si dispo.
- `COMET` : métrique neurale proche du jugement humain, si dispo.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Garde seulement les lignes evaluées
plot_df = eval_translation_df[eval_translation_df['status'] == 'ok'].copy()

if plot_df.empty:
    print('Aucune ligne evaluée (status=ok).')
else:
    model_order = ['marian', 'nllb', 'ollama', 'lstm']

    def _build_long(metric_prefix: str):
        """Fonction utilitaire pour  build long."""
        parts = []
        for m in model_order:
            c = f'{metric_prefix}_{m}'
            if c in plot_df.columns:
                sub = plot_df[['lang', c]].rename(columns={c: 'score'}).copy()
                sub['modele'] = m.upper()
                parts.append(sub)
        if not parts:
            return pd.DataFrame(columns=['lang', 'score', 'modele'])
        out = pd.concat(parts, ignore_index=True)
        out = out.dropna(subset=['score'])
        return out

    bleu_long = _build_long('bleu')
    chrf_long = _build_long('chrf')

    if bleu_long.empty and chrf_long.empty:
        print('Aucun score BLEU/chrF disponible pour le tracé.')
    else:
        langs = sorted(plot_df['lang'].astype(str).unique())
        fig_w = max(12, 2.2 * len(langs))
        fig, axes = plt.subplots(1, 2, figsize=(fig_w, 5.2))

        def _grouped_bar(ax, dfm, title):
            """Fonction utilitaire pour  grouped bar."""
            if dfm.empty:
                ax.text(0.5, 0.5, 'Pas de donnees', ha='center', va='center')
                ax.set_axis_off()
                return
            models = list(dfm['modele'].unique())
            langs_local = list(dfm['lang'].astype(str).unique())
            width = 0.8 / max(1, len(models))
            x = list(range(len(langs_local)))
            for j, model in enumerate(models):
                vals = []
                for lg in langs_local:
                    row = dfm[(dfm['lang'].astype(str) == lg) & (dfm['modele'] == model)]
                    vals.append(float(row['score'].iloc[0]) if len(row) else 0.0)
                offset = (j - (len(models) - 1) / 2.0) * width
                ax.bar([xi + offset for xi in x], vals, width=width, label=model)
            ax.set_xticks(x)
            ax.set_xticklabels(langs_local, rotation=30, ha='right')
            ax.set_title(title)
            ax.set_ylabel('Score')
            ax.grid(axis='y', alpha=0.25)
            ax.legend(title='Modele', fontsize=9)

        _grouped_bar(axes[0], bleu_long, 'BLEU par langue (Marian/NLLB/Ollama/LSTM)')
        _grouped_bar(axes[1], chrf_long, 'chrF par langue (Marian/NLLB/Ollama/LSTM)')
        plt.tight_layout()
        plt.show()


### **c) Analyse qualitative**

Comparer au moins 10 segments: sorties Marian vs NLLB, et typer les erreurs:
- contresens
- omission
- hallucination
- noms propres / ponctuation
- effet cascade d'erreurs ASR


In [ ]:
# Tri par debut pour eviter les lignes hors ordre temporel dans l'analyse qualitative.
base_qual = df_asr.sort_values(['start_sec', 'end_sec']).reset_index(drop=True).copy()

lang_show = COMPARE_LANGS[0] if len(COMPARE_LANGS) else TARGET_LANGS[0]
qual_cols = ["start_sec", "end_sec", "text_asr", f"text_marian_{lang_show}", f"text_nllb_{lang_show}"]

oll_col = f"text_ollama_{lang_show}"
if oll_col in base_qual.columns:
    qual_cols.append(oll_col)
if 'text_lstm_en' in base_qual.columns and lang_show == 'en':
    qual_cols.append('text_lstm_en')

# On evite d'afficher des lignes Ollama vides quand la colonne est presente.
if oll_col in base_qual.columns:
    base_qual = base_qual[(base_qual[oll_col].fillna("").str.strip().str.len() > 0)].copy()

n_show = max(10, min(20, len(base_qual))) if len(base_qual) > 0 else 0
qual_df = base_qual[qual_cols].head(n_show).copy() if n_show > 0 else pd.DataFrame(columns=qual_cols)

# Colonnes a remplir manuellement pendant l'analyse
qual_df["error_type"] = ""  # ex: contresens|omission|hallucination|noms_propres|ponctuation
qual_df["cascade_asr_impact"] = ""  # ex: faible/moyen/fort + commentaire

if len(qual_df) == 0:
    print("Pas de lignes qualitatives a afficher (Ollama vide sur cet echantillon).")
else:
    qual_df


### **d) Benchmark**

Cette section compare:
- temps total,
- latence moyenne (ms/segment),
- temps d'inference normalise par minute de video,
- memoire CPU (RAM) et memoire GPU (si CUDA dispo),
- accord Marian/NLLB (BLEU/chrF proxy).

`Ollama` est execute seulement si `ENABLE_OLLAMA=True` et serveur local accessible.


In [ ]:
import time
import pandas as pd

try:
    import psutil
except Exception:
    psutil = None

BENCH_LANGS = COMPARE_LANGS
BENCH_MAX_SEGMENTS = min(int(BENCH_MAX_SEGMENTS_CFG), len(df_asr))
bench_slice = df_asr.iloc[:BENCH_MAX_SEGMENTS].copy()
bench_texts = bench_slice["text_asr"].fillna("").tolist()
bench_duration_sec = float((bench_slice["end_sec"] - bench_slice["start_sec"]).clip(lower=0).sum())
bench_duration_min = max(bench_duration_sec / 60.0, 1e-9)

bench_results = {}
bench_rows = []

for lang in BENCH_LANGS:
    # Marian
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()
    t0 = time.perf_counter()
    marian_ok = True
    marian_err = ""
    marian_out = []
    try:
        marian_out = translate_with_marian(bench_texts, target_lang=lang)
    except Exception as e:
        marian_ok = False
        marian_err = str(e)
        marian_out = [""] * len(bench_texts)
    t1 = time.perf_counter()

    cpu_mb = round(psutil.Process().memory_info().rss / (1024**2), 2) if psutil else None
    gpu_mb = round(torch.cuda.max_memory_allocated() / (1024**2), 2) if torch.cuda.is_available() else None

    sec_total = t1 - t0
    bench_results[("marian", lang)] = marian_out
    bench_rows.append({
        "model": "marian",
        "lang": lang,
        "ok": marian_ok,
        "error": marian_err,
        "segments": len(bench_texts),
        "video_minutes": round(bench_duration_min, 3),
        "seconds_total": round(sec_total, 3),
        "ms_per_segment": round((sec_total * 1000) / max(1, len(bench_texts)), 2),
        "sec_per_video_min": round(sec_total / bench_duration_min, 3),
        "cpu_ram_mb": cpu_mb,
        "gpu_mem_mb": gpu_mb,
    })

    # NLLB
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()
    t0 = time.perf_counter()
    nllb_ok = True
    nllb_err = ""
    nllb_out = []
    try:
        nllb_out = translate_with_nllb(bench_texts, src_lang=SRC_LANG, tgt_lang=lang)
    except Exception as e:
        nllb_ok = False
        nllb_err = str(e)
        nllb_out = [""] * len(bench_texts)
    t1 = time.perf_counter()

    cpu_mb = round(psutil.Process().memory_info().rss / (1024**2), 2) if psutil else None
    gpu_mb = round(torch.cuda.max_memory_allocated() / (1024**2), 2) if torch.cuda.is_available() else None

    sec_total = t1 - t0
    bench_results[("nllb", lang)] = nllb_out
    bench_rows.append({
        "model": "nllb",
        "lang": lang,
        "ok": nllb_ok,
        "error": nllb_err,
        "segments": len(bench_texts),
        "video_minutes": round(bench_duration_min, 3),
        "seconds_total": round(sec_total, 3),
        "ms_per_segment": round((sec_total * 1000) / max(1, len(bench_texts)), 2),
        "sec_per_video_min": round(sec_total / bench_duration_min, 3),
        "cpu_ram_mb": cpu_mb,
        "gpu_mem_mb": gpu_mb,
    })

    # Ollama (optionnel)
    if ENABLE_OLLAMA:
        if torch.cuda.is_available():
            torch.cuda.reset_peak_memory_stats()
        t0 = time.perf_counter()
        ollama_ok = True
        ollama_err = ""
        ollama_out = []
        try:
            ollama_out = translate_with_ollama(bench_texts, src_lang=SRC_LANG, tgt_lang=lang)
        except Exception as e:
            ollama_ok = False
            ollama_err = str(e)
            ollama_out = [""] * len(bench_texts)
        t1 = time.perf_counter()

        cpu_mb = round(psutil.Process().memory_info().rss / (1024**2), 2) if psutil else None
        gpu_mb = round(torch.cuda.max_memory_allocated() / (1024**2), 2) if torch.cuda.is_available() else None

        sec_total = t1 - t0
        bench_results[("ollama", lang)] = ollama_out
        bench_rows.append({
            "model": "ollama",
            "lang": lang,
            "ok": ollama_ok,
            "error": ollama_err,
            "segments": len(bench_texts),
            "video_minutes": round(bench_duration_min, 3),
            "seconds_total": round(sec_total, 3),
            "ms_per_segment": round((sec_total * 1000) / max(1, len(bench_texts)), 2),
            "sec_per_video_min": round(sec_total / bench_duration_min, 3),
            "cpu_ram_mb": cpu_mb,
            "gpu_mem_mb": gpu_mb,
        })

    # LSTM (FR->EN uniquement), si la section LSTM a ete executee.
    if lang == 'en' and 'translate_with_lstm' in globals():
        if torch.cuda.is_available():
            torch.cuda.reset_peak_memory_stats()
        t0 = time.perf_counter()
        lstm_ok = True
        lstm_err = ""
        lstm_out = []
        try:
            lstm_out = translate_with_lstm(bench_texts)
        except Exception as e:
            lstm_ok = False
            lstm_err = str(e)
            lstm_out = [""] * len(bench_texts)
        t1 = time.perf_counter()

        cpu_mb = round(psutil.Process().memory_info().rss / (1024**2), 2) if psutil else None
        gpu_mb = round(torch.cuda.max_memory_allocated() / (1024**2), 2) if torch.cuda.is_available() else None

        sec_total = t1 - t0
        bench_results[("lstm", lang)] = lstm_out
        bench_rows.append({
            "model": "lstm",
            "lang": lang,
            "ok": lstm_ok,
            "error": lstm_err,
            "segments": len(bench_texts),
            "video_minutes": round(bench_duration_min, 3),
            "seconds_total": round(sec_total, 3),
            "ms_per_segment": round((sec_total * 1000) / max(1, len(bench_texts)), 2),
            "sec_per_video_min": round(sec_total / bench_duration_min, 3),
            "cpu_ram_mb": cpu_mb,
            "gpu_mem_mb": gpu_mb,
        })

bench_df = pd.DataFrame(bench_rows).sort_values(["lang", "model"]).reset_index(drop=True)
bench_df


In [ ]:
from sacrebleu import corpus_bleu, corpus_chrf

# Accord entre Marian (reference interne) et autres modeles disponibles.
compare_rows = []
for lang in BENCH_LANGS:
    marian_row = bench_df[(bench_df["lang"] == lang) & (bench_df["model"] == "marian")]
    if len(marian_row) == 0 or not bool(marian_row["ok"].iloc[0]):
        continue

    marian_out = bench_results.get(("marian", lang), [])
    for other in ["nllb", "ollama", "lstm"]:
        other_row = bench_df[(bench_df["lang"] == lang) & (bench_df["model"] == other)]
        if len(other_row) == 0 or not bool(other_row["ok"].iloc[0]):
            continue
        other_out = bench_results.get((other, lang), [])
        bleu = corpus_bleu(other_out, [marian_out]).score
        chrf = corpus_chrf(other_out, [marian_out]).score
        compare_rows.append({
            "lang": lang,
            "pair": f"MARIAN vs {other.upper()}",
            "agreement_bleu": round(bleu, 2),
            "agreement_chrf": round(chrf, 2),
        })

compare_df = pd.DataFrame(compare_rows).sort_values(["lang", "pair"]).reset_index(drop=True)
compare_df


### **e) Graphiques de comparaison**

Visualisations automatiques:
- temps total par modèle/langue,
- latence moyenne (ms/segment),
- RAM (proxy),
- accord Marian vs NLLB (BLEU/chrF proxy).

Les barplots incluent aussi `OLLAMA` si présent dans `bench_df`.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

try:
    import seaborn as sns
    _has_sns = True
except Exception:
    _has_sns = False

plot_df = bench_df.copy()
plot_df = plot_df[plot_df['ok'] == True].copy()
if plot_df.empty:
    print('bench_df ne contient pas de lignes valides (ok=True).')
else:
    plot_df["model"] = plot_df["model"].str.upper()

    if _has_sns:
        sns.set_theme(style="whitegrid", context="notebook")

    n_lang = max(1, plot_df['lang'].nunique())
    fig_w = max(16, 2.6 * n_lang)
    fig, axes = plt.subplots(2, 3, figsize=(fig_w, 10))

    def _bar(ax, y, title, ylabel, may_log=False):
        sub = plot_df.dropna(subset=[y]).copy()
        if sub.empty:
            ax.text(0.5, 0.5, 'Pas de donnees', ha='center', va='center')
            ax.set_axis_off()
            return

        if _has_sns:
            sns.barplot(data=sub, x="lang", y=y, hue="model", ax=ax, dodge=True)
        else:
            models = sorted(sub["model"].unique())
            langs = list(sub["lang"].astype(str).unique())
            width = 0.8 / max(1, len(models))
            x = list(range(len(langs)))
            for j, model in enumerate(models):
                sm = sub[sub["model"] == model]
                vals = []
                for lg in langs:
                    row = sm[sm["lang"].astype(str) == lg]
                    vals.append(float(row[y].iloc[0]) if len(row) else 0.0)
                offset = (j - (len(models) - 1) / 2.0) * width
                ax.bar([xi + offset for xi in x], vals, width=width, label=model)
            ax.set_xticks(x)
            ax.set_xticklabels(langs, rotation=30, ha='right')

        vals = sub[y].astype(float).values
        vmin = np.nanmin(vals) if len(vals) else 0
        vmax = np.nanmax(vals) if len(vals) else 0
        if may_log and vmin > 0 and vmax / max(vmin, 1e-9) >= 15:
            ax.set_yscale('log')
            ax.set_title(title + ' (echelle log)')
        else:
            ax.set_title(title)

        ax.set_xlabel("Langue cible")
        ax.set_ylabel(ylabel)
        ax.grid(axis='y', alpha=0.25)
        if len(sub['model'].unique()) > 0:
            ax.legend(title="Modele", fontsize=8)
        for lbl in ax.get_xticklabels():
            lbl.set_rotation(30)
            lbl.set_ha('right')

    _bar(axes[0, 0], "seconds_total", "Temps total inference", "secondes", may_log=True)
    _bar(axes[0, 1], "ms_per_segment", "Latence", "ms/segment", may_log=True)
    _bar(axes[0, 2], "sec_per_video_min", "Temps par minute video", "sec / min video", may_log=True)

    _bar(axes[1, 0], "cpu_ram_mb", "Memoire CPU (RAM)", "MB", may_log=False)
    _bar(axes[1, 1], "gpu_mem_mb", "Memoire GPU (pic)", "MB", may_log=False)

    # Accord Marian vs autres modeles: barres groupees pour meilleure lisibilite.
    if len(compare_df) > 0:
        cdf = compare_df.copy().dropna(subset=['agreement_bleu', 'agreement_chrf'], how='all')
        if len(cdf) > 0:
            labels = [f"{r['lang']}\n{r['pair'].replace('MARIAN vs ', '')}" for _, r in cdf.iterrows()]
            x = np.arange(len(labels))
            w = 0.38
            bleu_vals = cdf['agreement_bleu'].fillna(0).astype(float).values
            chrf_vals = cdf['agreement_chrf'].fillna(0).astype(float).values
            axes[1, 2].bar(x - w/2, bleu_vals, width=w, label='BLEU')
            axes[1, 2].bar(x + w/2, chrf_vals, width=w, label='chrF')
            axes[1, 2].set_xticks(x)
            axes[1, 2].set_xticklabels(labels, rotation=30, ha='right')
            axes[1, 2].set_title("Accord Marian vs autres")
            axes[1, 2].set_ylabel("Score")
            axes[1, 2].grid(axis='y', alpha=0.25)
            axes[1, 2].legend(fontsize=8)
        else:
            axes[1, 2].text(0.5, 0.5, "compare_df vide", ha="center", va="center")
            axes[1, 2].set_axis_off()
    else:
        axes[1, 2].text(0.5, 0.5, "compare_df vide", ha="center", va="center")
        axes[1, 2].set_axis_off()

    plt.tight_layout()
    plt.show()

    # Figure 2: couverture + matrice d'accord style "confusion matrix".
    model_names = sorted(plot_df['model'].str.lower().unique())

    cov_rows = []
    for m in model_names:
        vals = bench_results.get((m, 'en'), [])
        if len(vals) == 0:
            continue
        non_empty = sum(1 for x in vals if str(x).strip())
        cov_rows.append({'model': m.upper(), 'coverage_pct': 100.0 * non_empty / max(1, len(vals))})
    coverage_df = pd.DataFrame(cov_rows)

    # Matrice d'accord chrF pairwise entre sorties de modeles.
    avail = []
    for m in model_names:
        vals = bench_results.get((m, 'en'), [])
        if len(vals) > 0:
            avail.append(m)

    mat = pd.DataFrame(index=[m.upper() for m in avail], columns=[m.upper() for m in avail], dtype=float)
    from sacrebleu import corpus_chrf
    for a in avail:
        va = bench_results.get((a, 'en'), [])
        for b in avail:
            vb = bench_results.get((b, 'en'), [])
            pairs = [(x, y) for x, y in zip(va, vb) if str(x).strip() and str(y).strip()]
            if len(pairs) == 0:
                mat.loc[a.upper(), b.upper()] = np.nan
            else:
                ha = [p[0] for p in pairs]
                hb = [p[1] for p in pairs]
                mat.loc[a.upper(), b.upper()] = corpus_chrf(ha, [hb]).score

    fig2, ax2 = plt.subplots(1, 2, figsize=(14, 5))

    if len(coverage_df) > 0:
        if _has_sns:
            sns.barplot(data=coverage_df, x='model', y='coverage_pct', ax=ax2[0], palette='Set2')
        else:
            ax2[0].bar(coverage_df['model'], coverage_df['coverage_pct'])
        ax2[0].set_title('Couverture des sorties non vides')
        ax2[0].set_ylabel('% segments non vides')
        ax2[0].set_ylim(0, 105)
        ax2[0].grid(axis='y', alpha=0.25)
    else:
        ax2[0].text(0.5, 0.5, 'Coverage indisponible', ha='center', va='center')
        ax2[0].set_axis_off()

    if mat.size > 0:
        if _has_sns:
            sns.heatmap(mat, annot=True, fmt='.1f', cmap='mako', vmin=0, vmax=100, ax=ax2[1], cbar_kws={'label': 'chrF'})
        else:
            im = ax2[1].imshow(mat.values.astype(float), vmin=0, vmax=100, cmap='viridis')
            ax2[1].set_xticks(range(len(mat.columns)))
            ax2[1].set_xticklabels(mat.columns, rotation=45, ha='right')
            ax2[1].set_yticks(range(len(mat.index)))
            ax2[1].set_yticklabels(mat.index)
            fig2.colorbar(im, ax=ax2[1], label='chrF')
        ax2[1].set_title("Matrice d'accord pairwise (chrF)")
    else:
        ax2[1].text(0.5, 0.5, 'Matrice indisponible', ha='center', va='center')
        ax2[1].set_axis_off()

    plt.tight_layout()
    plt.show()


### **f) Analyse par conditions et entre differentes donnees**

Objectif demande:
- segments longs vs courts,
- debit rapide,
- vocabulaire technique,
- comparaison entre differents splits (`train`, `valid`, `test`).

Strategie:
- on construit des indicateurs sur les transcriptions de reference FR,
- on compare la distribution des conditions par split,
- on mesure la similarite Marian/NLLB par condition sur le split courant.


In [ ]:
import re
import pandas as pd
import matplotlib.pyplot as plt

try:
    import seaborn as sns
    _has_sns_split = True
except Exception:
    _has_sns_split = False

TECH_TERMS = {
    "algorithme", "modele", "donnees", "reseau", "neural", "apprentissage", "optimisation",
    "energie", "carbone", "biologie", "medecine", "finance", "systeme", "theorie"
}


def load_split_df(split_name: str) -> pd.DataFrame:
    return load_split_dataframe(DATA_ROOT, split_name, SRC_LANG)


def add_condition_columns(df: pd.DataFrame, text_col: str) -> pd.DataFrame:
    out = df.copy()
    out["duration_sec"] = (out["end_sec"] - out["start_sec"]).clip(lower=1e-6)
    out["char_len"] = out[text_col].fillna("").str.len()
    out["char_per_sec"] = out["char_len"] / out["duration_sec"]

    q_dur = out["duration_sec"].quantile(0.5)
    q_rate = out["char_per_sec"].quantile(0.75)

    out["cond_length"] = out["duration_sec"].apply(lambda x: "long" if x >= q_dur else "short")
    out["cond_rate"] = out["char_per_sec"].apply(lambda x: "fast" if x >= q_rate else "normal")

    def has_tech(txt: str) -> bool:
        toks = re.findall(r"[a-zA-ZÀ-ÿ]+", str(txt).lower())
        return any(t in TECH_TERMS for t in toks)

    out["cond_tech"] = out[text_col].apply(has_tech)
    return out


# A) Comparaison entre differents splits
split_rows = []
for sp in ["train", "valid", "test"]:
    try:
        d = load_split_df(sp)
    except Exception:
        continue
    d = add_condition_columns(d, "text_src")
    split_rows.append({
        "split": sp,
        "n_segments": len(d),
        "avg_duration_sec": round(float(d["duration_sec"].mean()), 3),
        "pct_long": round(float((d["cond_length"] == "long").mean()) * 100, 2),
        "pct_fast": round(float((d["cond_rate"] == "fast").mean()) * 100, 2),
        "pct_tech": round(float(d["cond_tech"].mean()) * 100, 2),
    })

split_condition_df = pd.DataFrame(split_rows)
display(split_condition_df)

if len(split_condition_df) > 0:
    metrics = ["avg_duration_sec", "pct_long", "pct_fast", "pct_tech"]
    heat = split_condition_df.set_index('split')[metrics]
    plt.figure(figsize=(8, 3.8))
    if _has_sns_split:
        sns.heatmap(heat, annot=True, fmt='.2f', cmap='crest')
    else:
        plt.imshow(heat.values, aspect='auto')
        plt.xticks(range(len(metrics)), metrics, rotation=25, ha='right')
        plt.yticks(range(len(heat.index)), heat.index)
    plt.title('Profil des splits (vue rapide)')
    plt.tight_layout()
    plt.show()


In [ ]:
# B) Qualite relative Marian vs autres modeles par condition (split courant)
cond_df = df_asr.copy()
cond_df["duration_sec"] = (cond_df["end_sec"] - cond_df["start_sec"]).clip(lower=1e-6)
cond_df["char_len"] = cond_df["text_asr"].fillna("").str.len()
cond_df["char_per_sec"] = cond_df["char_len"] / cond_df["duration_sec"]

q_dur = cond_df["duration_sec"].quantile(0.5)
q_rate = cond_df["char_per_sec"].quantile(0.75)
cond_df["cond_length"] = cond_df["duration_sec"].apply(lambda x: "long" if x >= q_dur else "short")
cond_df["cond_rate"] = cond_df["char_per_sec"].apply(lambda x: "fast" if x >= q_rate else "normal")


def has_tech(txt: str) -> bool:
    toks = re.findall(r"[a-zA-ZÀ-ÿ]+", str(txt).lower())
    return any(t in TECH_TERMS for t in toks)

cond_df["cond_tech"] = cond_df["text_asr"].apply(has_tech)

rows = []
for lang in COMPARE_LANGS:
    mar_col = f"text_marian_{lang}"
    if mar_col not in cond_df.columns:
        continue

    # Modeles compares a Marian pour cette langue.
    competitors = []
    nll_col = f"text_nllb_{lang}"
    oll_col = f"text_ollama_{lang}"
    if nll_col in cond_df.columns:
        competitors.append(("nllb", nll_col))
    if oll_col in cond_df.columns and cond_df[oll_col].fillna('').str.len().sum() > 0:
        competitors.append(("ollama", oll_col))
    if lang == 'en' and 'text_lstm_en' in cond_df.columns:
        competitors.append(("lstm", 'text_lstm_en'))

    for model_name, cmp_col in competitors:
        for cond_name, mask in [
            ("length=long", cond_df["cond_length"] == "long"),
            ("length=short", cond_df["cond_length"] == "short"),
            ("rate=fast", cond_df["cond_rate"] == "fast"),
            ("rate=normal", cond_df["cond_rate"] == "normal"),
            ("tech=true", cond_df["cond_tech"] == True),
            ("tech=false", cond_df["cond_tech"] == False),
        ]:
            sub = cond_df[mask]
            if len(sub) < 3:
                continue
            mar = sub[mar_col].fillna("").tolist()
            cmp_out = sub[cmp_col].fillna("").tolist()
            bleu = corpus_bleu(cmp_out, [mar]).score
            chrf = corpus_chrf(cmp_out, [mar]).score
            rows.append({
                "lang": lang,
                "model": model_name,
                "condition": cond_name,
                "n": len(sub),
                "agreement_bleu": round(bleu, 2),
                "agreement_chrf": round(chrf, 2),
            })

condition_agreement_df = pd.DataFrame(rows).sort_values(["lang", "model", "condition"]).reset_index(drop=True)
display(condition_agreement_df)

if len(condition_agreement_df) > 0:
    try:
        import seaborn as sns
        _has_sns_cond = True
    except Exception:
        _has_sns_cond = False

    piv = condition_agreement_df.pivot_table(index='condition', columns='model', values='agreement_chrf', aggfunc='mean')
    plt.figure(figsize=(8, 4.2))
    if _has_sns_cond:
        sns.heatmap(piv, annot=True, fmt='.1f', cmap='rocket_r', vmin=0, vmax=100)
    else:
        plt.imshow(piv.values, aspect='auto')
        plt.xticks(range(len(piv.columns)), piv.columns, rotation=30, ha='right')
        plt.yticks(range(len(piv.index)), piv.index)
    plt.title("Accord par condition (chrF, ref=Marian)")
    plt.tight_layout()
    plt.show()


## **<u>3) PIPELINE traduction automatique d'une vidéo</u>**

Cette section traite **explicitement** `data/inputs/test.mp4`:
1. extraction du texte (ASR, anglais),
2. traduction segment par segment vers le francais,
3. generation de `test.fr.srt`,
4. generation de la voix FR (`test.fr.tts.mp3`),
5. mux final en `outputs/test_trad.mp4` avec:
- piste audio 1: originale,
- piste audio 2: voix FR,
- piste sous-titres FR (mov_text).


In [ ]:
# Installation optionnelle (si besoin)
# %pip install -U edge-tts nest_asyncio

import asyncio
import subprocess
import time
from pathlib import Path

import pandas as pd
import torch
import edge_tts
from transformers import pipeline, AutoTokenizer, AutoModelForSeq2SeqLM
from IPython.display import Audio, Video, display

PROJECT_ROOT = Path('.').resolve() if (Path('.') / 'data').exists() else Path('..').resolve()
VIDEO_INPUT = PROJECT_ROOT / 'data' / 'inputs' / 'test.mp4'
OUTPUT_DIR = PROJECT_ROOT / 'outputs'
TMP_FR = OUTPUT_DIR / 'tmp_tts_fr'
TMP_ES = OUTPUT_DIR / 'tmp_tts_es'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
TMP_FR.mkdir(parents=True, exist_ok=True)
TMP_ES.mkdir(parents=True, exist_ok=True)

SRT_FR = OUTPUT_DIR / 'test.fr.srt'
TTS_FR = OUTPUT_DIR / 'test.fr.tts.aligned.m4a'
TTS_ES = OUTPUT_DIR / 'test.es.tts.aligned.m4a'
VIDEO_OUT = OUTPUT_DIR / 'test_trad.mp4'

assert VIDEO_INPUT.exists(), f"Video pas trouve : {VIDEO_INPUT}"


def fmt_srt(sec: float) -> str:
    h = int(sec // 3600)
    m = int((sec % 3600) // 60)
    s = int(sec % 60)
    ms = int(round((sec - int(sec)) * 1000))
    return f"{h:02}:{m:02}:{s:02},{ms:03}"


def probe_duration(path: Path) -> float:
    cmd = ['ffprobe', '-v', 'error', '-show_entries', 'format=duration', '-of', 'default=nw=1:nk=1', str(path)]
    return float(subprocess.check_output(cmd, text=True).strip())




def run_ffmpeg(cmd):
    """Lance ffmpeg et remonte stderr complet si echec (utile pour debug notebook)."""
    p = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
    if p.returncode != 0:
        err = (p.stderr or '').strip()
        short = '\n'.join(err.splitlines()[-60:])
        raise RuntimeError(f"ffmpeg a echoue (code={p.returncode})\n{short}")
    return p

def run_coro(coro):
    """Execute une coroutine meme dans un notebook deja en event loop."""
    try:
        loop = asyncio.get_running_loop()
    except RuntimeError:
        return asyncio.run(coro)
    import nest_asyncio
    nest_asyncio.apply(loop)
    return loop.run_until_complete(coro)


async def _synthesize_tts_once(text: str, voice: str, out_path: Path):
    comm = edge_tts.Communicate(text=text, voice=voice, rate='+5%')
    await comm.save(str(out_path))


def synthesize_tts_with_retry(text: str, voice: str, out_path: Path, retries: int = 6):
    """Genere une voix TTS avec retries en cas de coupure reseau."""
    last_err = None
    for k in range(retries):
        try:
            run_coro(_synthesize_tts_once(text, voice, out_path))
            return True
        except Exception as e:
            last_err = e
            wait_s = min(2 ** k, 20)
            print(f'[WARN] TTS retry {k+1}/{retries} failed for {out_path.name}: {e}. wait={wait_s}s')
            time.sleep(wait_s)
    print(f'[ERROR] TTS failed after retries for {out_path.name}: {last_err}')
    return False


def atempo_chain(factor: float) -> str:
    x = max(factor, 1e-6)
    vals = []
    while x > 2.0:
        vals.append(2.0)
        x /= 2.0
    while x < 0.5:
        vals.append(0.5)
        x /= 0.5
    vals.append(x)
    return ','.join([f'atempo={v:.6f}' for v in vals])


def group_words_to_segments(words, max_chars=75, max_dur=4.5, max_gap=0.35):
    """Regroupe les mots horodates en segments lisibles."""
    segs = []
    cur = None
    for st, en, txt in words:
        txt = txt.strip()
        if not txt:
            continue
        if cur is None:
            cur = {'start_sec': st, 'end_sec': en, 'text_en': txt}
            continue
        gap = st - cur['end_sec']
        cand_txt = (cur['text_en'] + ' ' + txt).strip()
        cand_dur = en - cur['start_sec']
        if len(cand_txt) <= max_chars and cand_dur <= max_dur and gap <= max_gap:
            cur['text_en'] = cand_txt
            cur['end_sec'] = en
        else:
            segs.append(cur)
            cur = {'start_sec': st, 'end_sec': en, 'text_en': txt}
    if cur is not None:
        segs.append(cur)
    out = pd.DataFrame(segs)
    if out.empty:
        return out
    min_gap = 0.05
    min_dur = 0.8
    out = out.sort_values(['start_sec', 'end_sec']).reset_index(drop=True)
    for i in range(len(out)):
        if i > 0:
            out.loc[i, 'start_sec'] = max(float(out.loc[i, 'start_sec']), float(out.loc[i - 1, 'end_sec']) + min_gap)
        out.loc[i, 'end_sec'] = max(float(out.loc[i, 'end_sec']), float(out.loc[i, 'start_sec']) + min_dur)
    return out


def translate_batch(texts, src='en', tgt='fr', batch_size=16):
    """Traduit en lot via le pipeline MT courant."""
    marian_candidates = {('en', 'fr'): ['Helsinki-NLP/opus-mt-en-fr'], ('en', 'es'): ['Helsinki-NLP/opus-mt-en-es']}.get((src, tgt), [])
    for model_name in marian_candidates:
        try:
            tok = AutoTokenizer.from_pretrained(model_name)
            mod = AutoModelForSeq2SeqLM.from_pretrained(model_name)
            mod.to('cuda' if torch.cuda.is_available() else 'cpu')
            outs = []
            for i in range(0, len(texts), batch_size):
                b = texts[i:i + batch_size]
                enc = tok(b, return_tensors='pt', padding=True, truncation=True)
                enc = {k: v.to(mod.device) for k, v in enc.items()}
                with torch.no_grad():
                    gen = mod.generate(**enc, max_new_tokens=128)
                outs.extend(tok.batch_decode(gen, skip_special_tokens=True))
            return outs
        except Exception:
            pass

    nllb = {'en': 'eng_Latn', 'fr': 'fra_Latn', 'es': 'spa_Latn'}
    tok = AutoTokenizer.from_pretrained('facebook/nllb-200-distilled-600M')
    mod = AutoModelForSeq2SeqLM.from_pretrained('facebook/nllb-200-distilled-600M')
    mod.to('cuda' if torch.cuda.is_available() else 'cpu')
    tok.src_lang = nllb[src]
    bos = tok.convert_tokens_to_ids(nllb[tgt])
    outs = []
    for i in range(0, len(texts), batch_size):
        b = texts[i:i + batch_size]
        enc = tok(b, return_tensors='pt', padding=True, truncation=True)
        enc = {k: v.to(mod.device) for k, v in enc.items()}
        with torch.no_grad():
            gen = mod.generate(**enc, forced_bos_token_id=bos, max_new_tokens=128)
        outs.extend(tok.batch_decode(gen, skip_special_tokens=True))
    return outs


def build_aligned_tts_track(sub_df: pd.DataFrame, text_col: str, voice: str, tmp_dir: Path, out_audio: Path):
    """Construit une piste TTS alignee sur les sous-titres segment par segment."""
    for f in tmp_dir.glob('*'):
        if f.is_file():
            f.unlink()

    aligned_inputs = []
    for i, r in sub_df.iterrows():
        txt = str(r[text_col]).strip()
        if not txt:
            continue

        start = float(r.start_sec)
        end = float(r.end_sec)
        slot = max(end - start, 0.35)

        raw = tmp_dir / f'raw_{i:04d}.mp3'
        adj = tmp_dir / f'adj_{i:04d}.m4a'

        ok = synthesize_tts_with_retry(txt, voice, raw)
        if not ok:
            # fallback silence segment
            subprocess.run(['ffmpeg','-y','-f','lavfi','-i','anullsrc=r=24000:cl=mono','-t',f'{slot:.3f}','-c:a','mp3', str(raw)], check=True)

        d_raw = probe_duration(raw)
        r_atempo = d_raw / slot if slot > 0 else 1.0
        chain = atempo_chain(r_atempo)

        cmd = ['ffmpeg','-y','-i', str(raw), '-filter:a', f'{chain},apad,atrim=0:{slot:.3f}', '-c:a', 'aac', str(adj)]
        run_ffmpeg(cmd)
        aligned_inputs.append((start, adj))

    if not aligned_inputs:
        raise RuntimeError(f'Aucun segment TTS genere pour {text_col}')

    video_duration = float(sub_df['end_sec'].max()) if len(sub_df) else 0.0
    ff_inputs = ['ffmpeg', '-y', '-f', 'lavfi', '-t', str(video_duration + 1.0), '-i', 'anullsrc=r=24000:cl=stereo']
    for _, f in aligned_inputs:
        ff_inputs += ['-i', str(f)]

    parts = []
    for i, (start, _) in enumerate(aligned_inputs):
        ms = int(round(start * 1000))
        parts.append(f'[{i+1}:a]adelay={ms}|{ms}[a{i}]')

    mix_inputs = '[0:a]' + ''.join([f'[a{i}]' for i in range(len(aligned_inputs))])
    parts.append(f'{mix_inputs}amix=inputs={len(aligned_inputs)+1}:duration=longest:dropout_transition=0[aout]')

    cmd = ff_inputs + ['-filter_complex', ';'.join(parts), '-map', '[aout]', '-c:a', 'aac', str(out_audio)]
    run_ffmpeg(cmd)


print('[1/7] ASR EN (timestamps mot-a-mot)...')
asr_pipe = pipeline(task='automatic-speech-recognition', model='openai/whisper-small', device=0 if torch.cuda.is_available() else -1)
asr_out = asr_pipe(str(VIDEO_INPUT), return_timestamps='word', chunk_length_s=20, generate_kwargs={'language': 'english'})

words = []
for ch in asr_out.get('chunks', []):
    ts = ch.get('timestamp', (None, None))
    txt = (ch.get('text', '') or '').strip()
    if not isinstance(ts, (tuple, list)) or len(ts) != 2:
        continue
    st, en = ts
    if st is None or en is None or not txt:
        continue
    words.append((float(st), float(en), txt))
if not words:
    raise RuntimeError('Aucun mot horodate detecte.')

seg_df = group_words_to_segments(words)
if seg_df.empty:
    raise RuntimeError('Aucun segment EN apres regroupement.')

video_duration = probe_duration(VIDEO_INPUT)
seg_df['start_sec'] = seg_df['start_sec'].clip(lower=0, upper=video_duration)
seg_df['end_sec'] = seg_df['end_sec'].clip(lower=0, upper=video_duration)
seg_df = seg_df[seg_df['end_sec'] > seg_df['start_sec']].reset_index(drop=True)
print('Segments:', len(seg_df))

print('[2/7] Traduction EN->FR et EN->ES...')
texts_en = seg_df['text_en'].tolist()
seg_df['text_fr'] = translate_batch(texts_en, src='en', tgt='fr')
seg_df['text_es'] = translate_batch(texts_en, src='en', tgt='es')

print('[3/7] SRT FR (timing conserve)...')
lines = []
for i, r in seg_df.iterrows():
    lines.append(str(i + 1))
    lines.append(f"{fmt_srt(float(r.start_sec))} --> {fmt_srt(float(r.end_sec))}")
    lines.append(str(r.text_fr).strip())
    lines.append('')
SRT_FR.write_text('\n'.join(lines), encoding='utf-8')
print('SRT FR:', SRT_FR)

print('[4/7] TTS FR aligne...')
build_aligned_tts_track(seg_df, 'text_fr', 'fr-FR-DeniseNeural', TMP_FR, TTS_FR)
print('TTS FR:', TTS_FR)

print('[5/7] TTS ES aligne...')
build_aligned_tts_track(seg_df, 'text_es', 'es-ES-ElviraNeural', TMP_ES, TTS_ES)
print('TTS ES:', TTS_ES)

print('[6/7] Mux final test_trad.mp4...')
cmd = [
    'ffmpeg', '-y', '-i', str(VIDEO_INPUT), '-i', str(TTS_FR), '-i', str(TTS_ES), '-sub_charenc', 'UTF-8', '-i', str(SRT_FR),
    '-filter_complex', '[1:a]apad[a_fr];[2:a]apad[a_es]',
    '-map', '0:v:0', '-map', '0:a:0', '-map', '[a_fr]', '-map', '[a_es]', '-map', '3:0',
    '-c:v', 'copy', '-c:a', 'aac', '-c:s', 'mov_text',
    '-metadata:s:a:0', 'title=Original Audio', '-metadata:s:a:0', 'language=eng',
    '-metadata:s:a:1', 'title=French TTS (Aligned)', '-metadata:s:a:1', 'language=fra',
    '-metadata:s:a:2', 'title=Spanish TTS (Aligned)', '-metadata:s:a:2', 'language=spa',
    '-metadata:s:s:0', 'title=French Subtitles', '-metadata:s:s:0', 'language=fra',
    '-disposition:a:0', '0', '-disposition:a:1', 'default', '-disposition:a:2', '0', '-disposition:s:0', 'default',
    '-t', str(video_duration), str(VIDEO_OUT)
]
run_ffmpeg(cmd)

print('[7/7] OK ->', VIDEO_OUT)
print('Si quelques segments TTS tombent en 503, fallback silence est applique localement.')

display(Audio(str(TTS_FR)))
display(Audio(str(TTS_ES)))
display(Video(str(VIDEO_OUT), embed=True, width=960))


## **<u>4) Conclusion</u>**


L’analyse comparative montre que les modèles basés sur des architectures Transformer, comme NLLB et Marian, donnent des résultats solides et exploitables dans un contexte réel. Les traductions produites par NLLB sont globalement très proches de celles de Marian, avec en plus une meilleure capacité à gérer plusieurs langues. En revanche, ce gain se fait au prix d’un modèle plus lourd et plus coûteux en ressources. En pratique, le choix entre les deux dépend donc du contexte d’utilisation et des contraintes matérielles : il faut trouver le bon équilibre entre qualité de traduction et coût de calcul.

Les résultats mettent également en évidence l’importance des contraintes opérationnelles. Les segments courts sont plus sensibles aux pertes de contexte, ce qui peut dégrader légèrement la traduction. De la même manière, un débit de parole élevé augmente le risque d’erreurs issues de l’ASR, qui se répercutent ensuite sur la traduction. On constate donc que la qualité finale ne dépend pas uniquement du modèle de traduction, mais de la solidité de l’ensemble du pipeline.

Ces éléments montrent qu’un système de traduction vidéo destiné à être utilisé en production ne peut pas être évalué uniquement à travers des métriques comme le BLEU. Il faut aussi prendre en compte la latence, la stabilité du système, la gestion des erreurs et la consommation de ressources. Ce sont ces aspects qui déterminent réellement la viabilité d’un déploiement.

En conclusion, le pipeline développé prouve qu’il est techniquement possible de mettre en place un système automatisé de sous-titrage et de doublage multilingue. Les modèles Transformer actuels sont suffisamment performants pour une intégration en conditions réelles, à condition de bien maîtriser le compromis entre qualité, temps d’inférence et contraintes matérielles.